# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from lightgbm import LGBMClassifier

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking2.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking2.parquet')

In [4]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,ridge_proba,voting_lgbm_cat_xgb_hist_proba,voting_lg_ridge_proba,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.519677,0.868510,0.664963,0.772948
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.395214,0.772514,0.543445,0.595178
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.259447,0.717012,0.409891,0.515461
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,0.005106,0.005494,0.003682,0.001281
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,0.150282,0.482416,0.283014,0.293768


In [5]:
X_test.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,ridge_proba,voting_lgbm_cat_xgb_hist_proba,voting_lg_ridge_proba,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_proba
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,0.004726,0.019849,0.005577,0.006190
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,0.033245,0.012173,0.048652,0.003070
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,0.003903,0.010304,0.004212,0.003695
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.296579,0.444558,0.477315,0.232534
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.548189,0.939648,0.695206,0.894775


# Machine Learning

In [6]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "objective": "binary",
        "n_estimators": trial.suggest_int("n_estimators", 30, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 4),
        "num_leaves": trial.suggest_int("num_leaves", 2, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 100, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 100, log=True),
        "verbosity": -1,
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LGBMClassifier(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=500, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-19 11:55:27,823] A new study created in memory with name: no-name-b1f1d6c3-5723-4bab-9318-1e486054d58a
Best trial: 7. Best value: 0.946324:   0%|▏                                                                                                               | 1/500 [00:10<1:24:23, 10.15s/it]

[I 2026-05-19 11:55:37,954] Trial 7 finished with value: 0.94632412344384 and parameters: {'n_estimators': 55, 'learning_rate': 0.01728958547161018, 'max_depth': 2, 'num_leaves': 16, 'min_child_samples': 87, 'subsample': 0.8637829805807256, 'colsample_bytree': 0.6992776552507046, 'reg_alpha': 3.7191745933425146e-05, 'reg_lambda': 0.00018940826559520394}. Best is trial 7 with value: 0.94632412344384.


Best trial: 6. Best value: 0.949667:   0%|▍                                                                                                                 | 2/500 [00:13<49:41,  5.99s/it]

[I 2026-05-19 11:55:41,036] Trial 6 finished with value: 0.9496667082764386 and parameters: {'n_estimators': 71, 'learning_rate': 0.044784103123623076, 'max_depth': 2, 'num_leaves': 15, 'min_child_samples': 71, 'subsample': 0.8426805756789664, 'colsample_bytree': 0.643698417386747, 'reg_alpha': 2.9804608740949465e-05, 'reg_lambda': 0.002221516692151999}. Best is trial 6 with value: 0.9496667082764386.


Best trial: 6. Best value: 0.949667:   1%|▋                                                                                                                 | 3/500 [00:15<37:14,  4.50s/it]

...

[I 2026-05-19 12:31:18,506] Trial 496 finished with value: 0.9498587792768755 and parameters: {'n_estimators': 269, 'learning_rate': 0.04494063170965147, 'max_depth': 4, 'num_leaves': 11, 'min_child_samples': 300, 'subsample': 0.9237020748038881, 'colsample_bytree': 0.6499276222856707, 'reg_alpha': 5.186829751801562, 'reg_lambda': 0.009387345721878142}. Best is trial 286 with value: 0.9498753425732973.


Best trial: 286. Best value: 0.949875: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████▌| 498/500 [35:50<00:03,  1.68s/it]

[I 2026-05-19 12:31:18,748] Trial 497 finished with value: 0.9498660440640055 and parameters: {'n_estimators': 270, 'learning_rate': 0.04457799568816225, 'max_depth': 4, 'num_leaves': 11, 'min_child_samples': 43, 'subsample': 0.9404941248844878, 'colsample_bytree': 0.6523962186108764, 'reg_alpha': 5.313988226694605, 'reg_lambda': 0.009470118202303361}. Best is trial 286 with value: 0.9498753425732973.


Best trial: 286. Best value: 0.949875: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [35:51<00:00,  4.30s/it]

[I 2026-05-19 12:31:18,979] Trial 498 finished with value: 0.9498628628231216 and parameters: {'n_estimators': 271, 'learning_rate': 0.04471282068563977, 'max_depth': 4, 'num_leaves': 11, 'min_child_samples': 41, 'subsample': 0.9968985715282146, 'colsample_bytree': 0.6482620052770925, 'reg_alpha': 5.017336723017805, 'reg_lambda': 0.07115848738449355}. Best is trial 286 with value: 0.9498753425732973.
[I 2026-05-19 12:31:19,170] Trial 499 finished with value: 0.9498645082322421 and parameters: {'n_estimators': 273, 'learning_rate': 0.044543423434898574, 'max_depth': 4, 'num_leaves': 11, 'min_child_samples': 41, 'subsample': 0.9957323589250318, 'colsample_bytree': 0.6508950729547044, 'reg_alpha': 5.380106408071916, 'reg_lambda': 0.07114107261577225}. Best is trial 286 with value: 0.9498753425732973.
Best trial score:
0.9498753425732973

Best params:
{'n_estimators': 264, 'learning_rate': 0.04988723993993441, 'max_depth': 4, 'num_leaves': 11, 'min_child_samples': 33, 'subsample': 0.937830

In [12]:
stacking_lgbm = LGBMClassifier(**study.best_params).fit(X_train, y_train.PitNextLap)

[LightGBM] [Info] Number of positive: 87381, number of negative: 351759
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001243 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 439140, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198982 -> initscore=-1.392668
[LightGBM] [Info] Start training from score -1.392668
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


# Submission

In [13]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [14]:
submission['PitNextLap'] = stacking_lgbm.predict_proba(X_test)[:, 1]

In [15]:
submission.to_csv('../data/submission/submission.csv', index=False)

In [16]:
X_train.columns

Index(['lgbm_proba', 'cat_proba', 'xgb_proba', 'hist_proba', 'lg_proba',
       'knn_proba', 'ridge_proba', 'voting_lgbm_cat_xgb_hist_proba',
       'voting_lg_ridge_proba',
       'stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_proba'],
      dtype='str')